In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [3]:
# File 1: Policyholder Information (50 Records)
policyholder_data = {
    'Policy_ID': [f'POL_{700+i}' for i in range(50)],
    'Driver_Age': [22, 45, 34, 60, 28, 50, 19, 41, 33, 55,
                   24, 48, 31, 62, 26, 52, 20, 43, 36, 58,
                   23, 46, 32, 61, 27, 51, 21, 42, 35, 56,
                   25, 49, 30, 63, 29, 53, 22, 44, 37, 59,
                   24, 47, 33, 64, 28, 54, 20, 40, 38, 57],
    'Credit_Score': [610, 750, 680, 810, 640, 780, 580, 720, 690, 790,
                     620, 760, 670, 820, 630, 770, 590, 730, 700, 800,
                     615, 755, 675, 815, 635, 775, 585, 725, 695, 795,
                     625, 765, 665, 825, 645, 785, 600, 735, 705, 805,
                     618, 758, 678, 818, 638, 778, 592, 722, 698, 788]
}

# File 2: Vehicle Risk Records (50 Records)
vehicle_data = {
    'Policy_ID': [f'POL_{700+i}' for i in range(50)],
    'Vehicle_Age_Yrs': [12, 3, 7, 1, 9, 2, 14, 4, 6, 2,
                        11, 2, 8, 1, 10, 3, 13, 5, 6, 1,
                        12, 3, 7, 1, 9, 2, 14, 4, 5, 2,
                        10, 2, 8, 1, 9, 3, 13, 4, 6, 1,
                        11, 3, 7, 1, 10, 2, 14, 5, 6, 2],
    'Annual_Mileage_KM': [22000, 11000, 15000, 8000, 18000, 9500, 25000, 12000, 14000, 9000,
                          21000, 10500, 16000, 7500, 19000, 9800, 24000, 12500, 14500, 8500,
                          22500, 11200, 15200, 8200, 18200, 9600, 25500, 12200, 13800, 9100,
                          20500, 10800, 16200, 7800, 18800, 10000, 24500, 12800, 14200, 8800,
                          21800, 11100, 15800, 8100, 18500, 9700, 25200, 12100, 14100, 9200]
}

# File 3: Claim History Target (50 Records)
claim_data = {
    'Policy_ID': [f'POL_{700+i}' for i in range(50)],
    'Made_Claim': [1, 0, 0, 0, 1, 0, 1, 0, 0, 0,
                   1, 0, 0, 0, 1, 0, 1, 0, 0, 0,
                   1, 0, 0, 0, 1, 0, 1, 0, 0, 0,
                   1, 0, 0, 0, 1, 0, 1, 0, 0, 0,
                   1, 0, 0, 0, 1, 0, 1, 0, 0, 0] # 15 Claims (30%), 35 No Claims (70%)
}

df_policy = pd.DataFrame(policyholder_data)
df_vehicle = pd.DataFrame(vehicle_data)
df_claim = pd.DataFrame(claim_data)

In [4]:
df_master = (
    df_policy
    .merge(df_vehicle, on='Policy_ID', how='inner')
    .merge(df_claim, on='Policy_ID', how='inner')
)

In [5]:
print("Master Dataset Shape:", df_master.shape)

print("\nMaster Insurance Dataset:")
display(df_master.head())

Master Dataset Shape: (50, 6)

Master Insurance Dataset:


,Policy_ID,Driver_Age,Credit_Score,Vehicle_Age_Yrs,Annual_Mileage_KM,Made_Claim
0,POL_700,22,610,12,22000,1
1,POL_701,45,750,3,11000,0
2,POL_702,34,680,7,15000,0
3,POL_703,60,810,1,8000,0
4,POL_704,28,640,9,18000,1


In [6]:
X = df_master[
    [
        'Driver_Age',
        'Credit_Score',
        'Vehicle_Age_Yrs',
        'Annual_Mileage_KM'
    ]
]

y = df_master['Made_Claim']

In [7]:
full_positive_percentage = y.mean() * 100

print(f"Full Dataset Claim Rate: {full_positive_percentage:.1f}%")

Full Dataset Claim Rate: 30.0%


In [8]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [9]:
# ================================================================
# PROJECT 3 — CROSS-VALIDATION & OVERFITTING DIAGNOSTICS PIPELINE
# Platform: Google Colab
# Language: Python
# Libraries: Pandas, NumPy, Scikit-Learn
# ================================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


# ================================================================
# INPUT DATA
# ================================================================

# File 1: Policyholder Information (50 Records)
policyholder_data = {
    "Policy_ID": [f"POL_{700+i}" for i in range(50)],
    "Driver_Age": [
        22, 45, 34, 60, 28, 50, 19, 41, 33, 55,
        24, 48, 31, 62, 26, 52, 20, 43, 36, 58,
        23, 46, 32, 61, 27, 51, 21, 42, 35, 56,
        25, 49, 30, 63, 29, 53, 22, 44, 37, 59,
        24, 47, 33, 64, 28, 54, 20, 40, 38, 57
    ],
    "Credit_Score": [
        610, 750, 680, 810, 640, 780, 580, 720, 690, 790,
        620, 760, 670, 820, 630, 770, 590, 730, 700, 800,
        615, 755, 675, 815, 635, 775, 585, 725, 695, 795,
        625, 765, 665, 825, 645, 785, 600, 735, 705, 805,
        618, 758, 678, 818, 638, 778, 592, 722, 698, 788
    ]
}

# File 2: Vehicle Risk Records (50 Records)
vehicle_data = {
    "Policy_ID": [f"POL_{700+i}" for i in range(50)],
    "Vehicle_Age_Yrs": [
        12, 3, 7, 1, 9, 2, 14, 4, 6, 2,
        11, 2, 8, 1, 10, 3, 13, 5, 6, 1,
        12, 3, 7, 1, 9, 2, 14, 4, 5, 2,
        10, 2, 8, 1, 9, 3, 13, 4, 6, 1,
        11, 3, 7, 1, 10, 2, 14, 5, 6, 2
    ],
    "Annual_Mileage_KM": [
        22000, 11000, 15000, 8000, 18000, 9500, 25000, 12000, 14000, 9000,
        21000, 10500, 16000, 7500, 19000, 9800, 24000, 12500, 14500, 8500,
        22500, 11200, 15200, 8200, 18200, 9600, 25500, 12200, 13800, 9100,
        20500, 10800, 16200, 7800, 18800, 10000, 24500, 12800, 14200, 8800,
        21800, 11100, 15800, 8100, 18500, 9700, 25200, 12100, 14100, 9200
    ]
}

# File 3: Claim History Target (50 Records)
claim_data = {
    "Policy_ID": [f"POL_{700+i}" for i in range(50)],
    "Made_Claim": [
        1, 0, 0, 0, 1, 0, 1, 0, 0, 0,
        1, 0, 0, 0, 1, 0, 1, 0, 0, 0,
        1, 0, 0, 0, 1, 0, 1, 0, 0, 0,
        1, 0, 0, 0, 1, 0, 1, 0, 0, 0,
        1, 0, 0, 0, 1, 0, 1, 0, 0, 0
    ]
}

df_policy = pd.DataFrame(policyholder_data)
df_vehicle = pd.DataFrame(vehicle_data)
df_claim = pd.DataFrame(claim_data)


# ================================================================
# TASK 1 — RELATIONAL INSURANCE DATA MERGING
# ================================================================

# Start with policyholder information.
# Inner merge means only Policy_ID values present in BOTH tables survive.
df_master = pd.merge(
    df_policy,
    df_vehicle,
    on="Policy_ID",
    how="inner"
)

# Add claim history using the same primary key.
df_master = pd.merge(
    df_master,
    df_claim,
    on="Policy_ID",
    how="inner"
)

print("\n========== TASK 1: MASTER DATASET ==========\n")
print(df_master.head())

print("\nMaster Dataset Shape:", df_master.shape)
print("Expected: 50 rows and 6 columns")

# Safety checks for the relational join.
assert len(df_master) == 50, "Master dataset should contain 50 records."
assert df_master["Policy_ID"].is_unique, "Policy_ID should be unique."
assert df_master["Made_Claim"].notna().all(), "Target should not contain missing values."


# ================================================================
# FEATURE / TARGET SEPARATION
# ================================================================

features = [
    "Driver_Age",
    "Credit_Score",
    "Vehicle_Age_Yrs",
    "Annual_Mileage_KM"
]

target = "Made_Claim"

X = df_master[features]
y = df_master[target]

print("\nFeatures (X):", ", ".join(features))
print("Target (y):", target)

positive_rate = y.mean() * 100

print(f"Overall Positive Claim Rate: {positive_rate:.1f}%")


# ================================================================
# TASK 2 — STANDARD 5-FOLD CROSS-VALIDATION
# ================================================================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("\n========== TASK 2: STANDARD K-FOLD SUMMARY ==========\n")

random_fold_rates = []

for fold, (train_index, val_index) in enumerate(
    kf.split(X),
    start=1
):
    # train_index contains row positions assigned to training.
    # val_index contains row positions assigned to validation.
    y_train = y.iloc[train_index]
    y_val = y.iloc[val_index]

    train_claims = int(y_train.sum())
    val_claims = int(y_val.sum())

    train_rate = y_train.mean() * 100
    val_rate = y_val.mean() * 100

    random_fold_rates.append(val_rate)

    print(
        f"Fold {fold} | "
        f"Train: {len(train_index)} Records | "
        f"Validation: {len(val_index)} Records | "
        f"Validation Claims: {val_claims} "
        f"({val_rate:.1f}%) | "
        f"Train Claims: {train_claims} "
        f"({train_rate:.1f}%)"
    )

print(
    f"\nStandard K-Fold Validation Positive Rate Range: "
    f"{min(random_fold_rates):.1f}% - {max(random_fold_rates):.1f}%"
)

print(
    f"Standard K-Fold Validation Rate Standard Deviation: "
    f"{np.std(random_fold_rates, ddof=0):.2f}%"
)


# ================================================================
# TASK 3 — STRATIFIED 5-FOLD CROSS-VALIDATION
# ================================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("\n========== TASK 3: STRATIFIED K-FOLD SUMMARY ==========\n")

stratified_fold_rates = []

for fold, (train_index, val_index) in enumerate(
    skf.split(X, y),
    start=1
):
    y_train = y.iloc[train_index]
    y_val = y.iloc[val_index]

    train_claims = int(y_train.sum())
    val_claims = int(y_val.sum())

    train_rate = y_train.mean() * 100
    val_rate = y_val.mean() * 100

    stratified_fold_rates.append(val_rate)

    print(
        f"Fold {fold} | "
        f"Train: {len(train_index)} Records | "
        f"Validation: {len(val_index)} Records | "
        f"Validation Claims: {val_claims} "
        f"({val_rate:.1f}%) | "
        f"Train Claims: {train_claims} "
        f"({train_rate:.1f}%)"
    )

print(
    f"\nStratified K-Fold Validation Positive Rate Range: "
    f"{min(stratified_fold_rates):.1f}% - {max(stratified_fold_rates):.1f}%"
)

print(
    f"Stratified K-Fold Validation Rate Standard Deviation: "
    f"{np.std(stratified_fold_rates, ddof=0):.2f}%"
)


# ================================================================
# TASK 4 — BASELINE LOGISTIC REGRESSION CROSS-VALIDATION
# ================================================================

print("\n========== TASK 4: LOGISTIC REGRESSION CV ==========\n")

# Scaling is included because the features have very different
# numeric ranges:
# Driver_Age           -> around tens
# Credit_Score         -> hundreds
# Annual_Mileage_KM    -> thousands
#
# Pipeline is important because the scaler is fitted separately
# inside every training fold. This prevents validation-data leakage.

model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("logistic_regression", LogisticRegression(max_iter=1000))
    ]
)

accuracy_scores = []

for fold, (train_index, val_index) in enumerate(
    skf.split(X, y),
    start=1
):
    X_train = X.iloc[train_index]
    X_val = X.iloc[val_index]

    y_train = y.iloc[train_index]
    y_val = y.iloc[val_index]

    # Fit only on the current training fold.
    model.fit(X_train, y_train)

    # Predict only the current validation fold.
    y_pred = model.predict(X_val)

    fold_accuracy = accuracy_score(y_val, y_pred)

    accuracy_scores.append(fold_accuracy)

    print(
        f"Fold {fold} Accuracy: "
        f"{fold_accuracy * 100:.1f}%"
    )

mean_accuracy = np.mean(accuracy_scores)
std_accuracy = np.std(accuracy_scores, ddof=0)

print(
    f"\nMean Validation Accuracy: "
    f"{mean_accuracy * 100:.1f}% "
    f"(+/- {std_accuracy * 100:.1f}%)"
)


# ================================================================
# TASK 5 — OVERFITTING & UNDERFITTING CONCEPTUAL ANALYSIS
# ================================================================

print("\n========== TASK 5: DIAGNOSTIC MATRIX ==========\n")

diagnostic_matrix = pd.DataFrame({
    "Scenario": [
        "Underfitting",
        "Overfitting",
        "Optimal Model"
    ],
    "Training Error": [
        "High",
        "Very Low",
        "Low"
    ],
    "Validation Error": [
        "High",
        "High",
        "Low"
    ],
    "Interpretation": [
        "Model is too simple and fails to capture important patterns.",
        "Model fits training data too closely and does not generalize well.",
        "Model captures useful patterns and generalizes well."
    ]
})

print(diagnostic_matrix.to_string(index=False))


# ================================================================
# FINAL SUMMARY
# ================================================================

print("\n")
print("=" * 70)
print("CROSS-VALIDATION & OVERFITTING DIAGNOSTICS")
print("=" * 70)

print(f"\nMaster Insurance Dataset Records : {len(df_master)}")
print(
    "Features (X)                     : "
    + ", ".join(features)
)
print(
    f"Target (y)                       : {target} "
    f"({positive_rate:.1f}% Positive Rate)"
)

print("\nCross-Validation Comparison:")

print(
    f"- Standard K-Fold (5 Splits)     : "
    f"Validation positive rate: "
    f"{min(random_fold_rates):.1f}% - "
    f"{max(random_fold_rates):.1f}%"
)

print(
    f"- Stratified K-Fold (5 Splits)   : "
    f"Validation positive rate: "
    f"{min(stratified_fold_rates):.1f}% - "
    f"{max(stratified_fold_rates):.1f}%"
)

print(
    f"\nBaseline Evaluation Results:"
    f"\n- Mean CV Accuracy Score         : "
    f"{mean_accuracy * 100:.1f}% "
    f"(+/- {std_accuracy * 100:.1f}%)"
)

print("\nConclusion:")
print(
    "Stratified K-Fold maintains a similar class distribution in each "
    "validation fold, making model evaluation more stable when the "
    "target classes are imbalanced."
)


========== TASK 1: MASTER DATASET ==========

  Policy_ID  Driver_Age  Credit_Score  Vehicle_Age_Yrs  Annual_Mileage_KM  \
0   POL_700          22           610               12              22000   
1   POL_701          45           750                3              11000   
2   POL_702          34           680                7              15000   
3   POL_703          60           810                1               8000   
4   POL_704          28           640                9              18000   

   Made_Claim  
0           1  
1           0  
2           0  
3           0  
4           1  

Master Dataset Shape: (50, 6)
Expected: 50 rows and 6 columns

Features (X): Driver_Age, Credit_Score, Vehicle_Age_Yrs, Annual_Mileage_KM
Target (y): Made_Claim
Overall Positive Claim Rate: 30.0%

========== TASK 2: STANDARD K-FOLD SUMMARY ==========

Fold 1 | Train: 40 Records | Validation: 10 Records | Validation Claims: 2 (20.0%) | Train Claims: 13 (32.5%)
Fold 2 | Train: 40 Records | Va